In [0]:
import pandas as pd 
from pyspark.sql import SparkSession

In [0]:
df = pd.read_csv("https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data/sales.csv")

display(df)

In [0]:
spark = SparkSession.builder.getOrCreate()
spark_df = spark.createDataFrame(df)


In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json

# Sample one JSON string from the 'product' column
sample_json = (
    spark_df.select("product")
    .filter(col("product").isNotNull())
    .first()[0]
)

# Infer schema from sample JSON
json_schema = schema_of_json(sample_json)

# Flatten JSON column
df_flat = (
    spark_df.withColumn(
        "product_json",
        from_json(col("product"), json_schema)
    )
    .select("*", "product_json.*")
    .drop("product", "product_json")
)

display(df_flat)

In [0]:
df_flat.write.format("delta").mode("overwrite").saveAsTable("batch.raw_data.sales")